In [1]:
# Setup

%load_ext autoreload
%autoreload 2

import json
import logging
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel

from comments import prompts
from comments.llm_generator import LLMGenerator
from comments.llm_labeling import LLMLabeler
from comments.static_labeling import label_unfinished_comments, label_code_snippets_comments
from shared import DifferencesEvaluator, load_dataset, save_dataset, ManualLabeler, add_filename_suffix
from shared.deduplicate import JsonDeduplicator
from shared.llm_connector import Model
from shared.utils import map_labels, dataset_remove_properties, find_by_key
from shared.sampling import random_undersample
from datasets import load_dataset as load_dataset_hf, concatenate_datasets, load_from_disk

load_dotenv()

logging.basicConfig(level=logging.INFO)
deduplicated_file_path = None

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Deduplication

input_path = "/media/martin/Big data1/datasets/github-projects/hive.json"
deduplicator = JsonDeduplicator()
file_path = Path(input_path)

deduplicated_file_path = deduplicator.deduplicate_dataset_file(file_path)

In [ ]:
# Labeling unfinished comments - LLM

UNFINISHED_COMMENT_ATTR = "unfinished_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

class ResponseModel(BaseModel):
    is_unfinished: bool

def process_response(response: ResponseModel, element: dict[str, Any]):
    element[UNFINISHED_COMMENT_ATTR] = 1 if response.is_unfinished else 0

labeler = LLMLabeler[ResponseModel](init_prompt=prompts.TODO_COMMENTS_INIT_PROMPT, run_prompt=prompts.RUN_PROMPT_1,
                                    labeled_attribute=UNFINISHED_COMMENT_ATTR, model=Model.LLAMA_3_1,
                                    process_response=process_response, response_class=ResponseModel)
await labeler.label_dataset(deduplicated_file_path, "-llama-labeled-unfinished")


In [ ]:
# Labeling unfinished comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_unfinished_comments(deduplicated_file_path)


In [ ]:
# Labeling unfinished comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="unfinished_comment_label", to_compare_keys=["unfinished_comment_llm", "unfinished_comment_static"], no_conflict_key="unfinished_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Labeling commented code comments - LLM

CODE_COMMENT_ATTR = "code_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

class ResponseModel(BaseModel):
    is_code_comment: bool

def process_response(response: ResponseModel, element: dict[str, Any]):
    element[CODE_COMMENT_ATTR] = 1 if response.is_code_comment else 0


labeler = LLMLabeler[ResponseModel](init_prompt=prompts.CODE_COMMENTS_INIT_PROMPT_2, run_prompt=prompts.RUN_PROMPT_1,
                                    labeled_attribute=CODE_COMMENT_ATTR, model=Model.LLAMA_3_1,
                                    process_response=process_response, response_class=ResponseModel)
await labeler.label_dataset(deduplicated_file_path, "-llm-labeled")

In [ ]:
# Labeling commented code comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_code_snippets_comments(deduplicated_file_path)

In [ ]:
# Labeling commented code comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="code_comment_label", to_compare_keys=["code_comment_llm", "code_comment_static"], no_conflict_key="code_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Map LLM a static labels to final label

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

mapping = {
    "code_comment_static": "code_comment",
    "unfinished_comment_static": "technical_debt"
}

dataset = load_dataset(deduplicated_file_path)
dataset = map_labels(mapping, "clean", dataset)

save_dataset(deduplicated_file_path, dataset)


In [ ]:
# Remove unused dataset properties

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

dataset = load_dataset(deduplicated_file_path)
dataset_remove_properties(["unfinished_comment_llm", "unfinished_comment_static", "unfinished_comment_label", "code_comment_llm", "code_comment_static", "code_comment_label"], dataset)

save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Manual labeling

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

# labeler = ManualLabeler(lambda el: el["unfinished_comment_static"] == 1, "unfinished_comment_static")
labeler = ManualLabeler(lambda el: el["code_comment_static"] == 1, "code_comment_static")
labeler.evaluate_labeled(deduplicated_file_path, )

In [ ]:
# LLM - generate TODO comments

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

generator = LLMGenerator(prompts.GENERATE_VARIATIONS_TODO_2, deduplicated_file_path, "todo-generated.json")

def filter_fnc(element):
    return "technical_debt" in element["labels"]

await generator.for_each_element(filter_fnc)

In [ ]:
# LLM - generate code comments

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

generator = LLMGenerator(prompts.GENERATE_VARIATIONS_CODE, deduplicated_file_path, "code-comments-generated.json")

def filter_fnc(element):
    return "code_comment" in element["labels"]

await generator.for_each_element(filter_fnc)

In [ ]:
# Merge generated into the dataset

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

folder_path = "/media/martin/Big data1/datasets/github-projects/"
# generated_files = ["tomcat-deduplicated-0-todo-generated.json", "tomcat-deduplicated-1-todo-generated.json",
#                    "tomcat-deduplicated-2-todo-generated.json", "tomcat-deduplicated-3-todo-generated.json",
#                    "tomcat-deduplicated-4-todo-generated.json"]

generated_files = ["tomcat-deduplicated-0-code-comments-generated.json", "tomcat-deduplicated-1-code-comments-generated.json",
                   "tomcat-deduplicated-2-code-comments-generated.json", "tomcat-deduplicated-3-code-comments-generated.json",
                   "tomcat-deduplicated-4-code-comments-generated.json"]

# generated_files = ["tomcat-deduplicated-1-todo-generated.json"]
dataset = load_dataset(deduplicated_file_path)
print(len(dataset))

def process_file(p: Path):
    with open(p, "r") as f:
        content = json.load(f)

    for element in content:
        original_value = find_by_key(dataset, "text", element["origin"])

        for generated_text in element["generated"]:
            new_value = original_value.copy()
            new_value["text"] = generated_text
            dataset.append(new_value)

for file in generated_files:
    path = Path(folder_path + file)
    process_file(path)

print(len(dataset))
save_dataset(deduplicated_file_path, dataset)


In [ ]:
if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))


print(len(list(filter(lambda el : el["unfinished_comment_static"] == 1, load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/hive.json"))))))
print(len(list(filter(lambda el : "technical_debt" in el["labels"], load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/pa165-projects.json"))))))
print(len(list(filter(lambda el : "technical_debt" in el["labels"], load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/thingsboard.json"))))))
print(len(list(filter(lambda el : "technical_debt" in el["labels"], load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/tomcat.json"))))))

print(len(list(filter(lambda el : el["code_comment_static"] == 1, load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/hive.json"))))))
print(len(list(filter(lambda el : "code_comment" in el["labels"], load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/pa165-projects.json"))))))
print(len(list(filter(lambda el : "code_comment" in el["labels"], load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/thingsboard.json"))))))
print(len(list(filter(lambda el : "code_comment" in el["labels"], load_dataset(Path("/media/martin/Big data1/datasets/Jena-datasets/comments/tomcat.json"))))))

In [ ]:
# Clean and convert to the arrow format

dataset_pa165 = load_dataset_hf("json", data_files="/media/martin/Big data1/datasets/Jena-datasets/comments/pa165-projects.json")
dataset_hive = load_dataset_hf("json", data_files="/media/martin/Big data1/datasets/Jena-datasets/comments/hive.json")
dataset_thingsboard = load_dataset_hf("json", data_files="/media/martin/Big data1/datasets/Jena-datasets/comments/thingsboard.json")
dataset_tomcat = load_dataset_hf("json", data_files="/media/martin/Big data1/datasets/Jena-datasets/comments/tomcat.json")

dataset = concatenate_datasets([dataset_pa165["train"], dataset_hive["train"], dataset_thingsboard["train"], dataset_tomcat["train"]])
dataset.save_to_disk("/media/martin/Big data1/datasets/Jena-datasets/comments/dataset")

def clean_data(data):
    text = data["text"]

    if text.startswith("*"):
        data["text"] = text.removeprefix("*").strip()

    if text.startswith("//"):
        data["text"] = text.removeprefix("//").strip()

    return data


cleaned = dataset.map(lambda data: clean_data(data))
cleaned.save_to_disk("/media/martin/Big data1/datasets/Jena-datasets/comments/dataset")


In [ ]:
# Random undersampling of imbalanced classes

from collections import Counter

path = input("Insert dataset path: ")
dataset = load_from_disk(path, keep_in_memory=False)

undersampled = random_undersample(dataset, "clean", 3500)

all_labels = [label for item in undersampled for label in item['labels']]
label_counts = Counter(all_labels)
print(label_counts)

new_path = add_filename_suffix(Path(path), "-undersampled")
undersampled.save_to_disk(new_path)


In [ ]:
path = input("Insert dataset path: ")
dataset = load_from_disk(path, keep_in_memory=False)

splitted = dataset.train_test_split(test_size=0.2)

new_path = add_filename_suffix(Path(path), "-splitted")
splitted.save_to_disk(new_path)

In [2]:
# Train

from comments.train import CommentsTrainer

path = input("Insert dataset path: ")
output_path = input("Insert output path: ")
classes = ["clean", "code_comment", "technical_debt"]
special_tokens = ["JAVADOC", "LINE", "BLOCK"]

trainer = CommentsTrainer(output_path, classes, special_tokens)

dataset = load_from_disk(path, keep_in_memory=False)
trainer.train_model(dataset)

INFO:CommentsTrainer:Initializing tokenizer...
INFO:CommentsTrainer:Preprocessing dataset...
/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce MX350 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce MX350 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch

AcceleratorError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
